# IMF Country Report → LLM → Dashboard Pipeline
**Country:** Kenya · **Period:** 2024–2025
**Report:** IMF Country Report No. 24/13 — Kenya: 2023 Article IV Consultation, Sixth Reviews under the EFF/ECF (published January 2024)

**How to run**
1. Get a free Gemini API key: https://aistudio.google.com/apikey
2. In Colab, open the 🔑 **Secrets** panel → add `GEMINI_API_KEY` → enable notebook access.
3. **Runtime → Run all.** The PDF downloads automatically; everything is saved to `outputs/` and `figures/`.

**Pipeline**
`PDF → clean text → sentence-aware chunks → LLM 1 (extract / summarise) → LLM 2 (judge) → JSON files → FAISS RAG → pd.melt → dashboard`

**Design notes**
- All LLM calls go through one function `ask_llm()` with `temperature=0` and a **local disk cache** (`cache/`), so re-runs are free and repeatable.
- Two different models are used so the judge does not mark its own work.
- No secret keys are stored in the notebook; no local file paths are hard-coded.

## 0. Setup

In [ ]:
%pip install -q pypdf tiktoken google-genai sentence-transformers faiss-cpu

In [ ]:
import os, re, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
os.makedirs("outputs", exist_ok=True)
os.makedirs("figures", exist_ok=True)

In [ ]:
# API key - never typed into the code. Checked in this order:
# 1) environment variable GEMINI_API_KEY (any computer)
# 2) Colab 🔑 Secrets panel
# No key? The notebook still runs: every answer comes from the saved cache
# (gemini_responses.json in the GitHub repo), so the marker needs no key.
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = None

from google import genai
client = genai.Client(api_key=API_KEY) if API_KEY else None
print("API key found" if client else "No API key - using saved answers from the cache")

# IMPORTANT: the cache key includes the model name. These MUST be the exact names
# that produced gemini_responses.json, otherwise every call is a cache miss and the
# marker's run will fail without a key. (Your 429 log mentions "gemini-3.6-flash" -
# check which name you actually used and set it here.)
EXTRACT_MODEL = "gemini-3.5-flash"       # LLM 1: extraction / summarisation
JUDGE_MODEL   = "gemini-3.5-flash-lite"  # LLM 2: evaluator (a different model, so it never marks its own work)

## Task 1 — PDF processing & summarisation
### 1.1 Load the PDF

In [ ]:
# The PDF is fetched automatically - no manual upload needed.
# Sources are tried in order until one works:
GITHUB_USER = "ParimiVenkatesh"        # <- your GitHub username
REPO        = "imf-kenya-llm-pipeline"
PDF_PATH    = "kenya_imf_report.pdf"
RAW = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
PDF_SOURCES = [
    f"{RAW}/{PDF_PATH}",                                                                    # copy in this repo (IMF blocks Colab)
    "https://www.imf.org/-/media/Files/Publications/CR/2024/English/1KENEA2024001.ashx",   # IMF official
]

import requests

def download_pdf(url, path):
    """Download the report once; reuse the local copy on re-runs (disk caching)."""
    if os.path.exists(path):
        return True
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        if r.content[:4] == b"%PDF":          # check it is a real PDF, not an error page
            open(path, "wb").write(r.content)
            return True
    except Exception as e:
        print("Download error:", e)
    return False

ok = any(download_pdf(u, PDF_PATH) for u in PDF_SOURCES)   # stops at the first success
if not ok:
    raise FileNotFoundError("Could not download the report from any source - check internet "
                            "and that GITHUB_USER is set correctly.")
print("Using:", PDF_PATH, f"({os.path.getsize(PDF_PATH)//1024} KB)")

In [ ]:
from pypdf import PdfReader
reader = PdfReader(PDF_PATH)
pages = [p.extract_text() or "" for p in reader.pages]
print("Pages:", len(pages))
print(pages[5][:800])

### 1.2 Clean the text

In [ ]:
from collections import Counter

# Lines that repeat on many pages = running headers/footers
line_counts = Counter()
for p in pages:
    for line in set(l.strip() for l in p.split("\n")):
        if line:
            line_counts[line] += 1
# A line is a header/footer if it is short and appears on >30% of pages
repeated = {l for l, c in line_counts.items() if c > len(pages) * 0.3 and len(l) < 60}
print("Header/footer lines removed:", list(repeated)[:10])

In [ ]:
def clean_page(text):
    lines = []
    for l in text.split("\n"):
        l = l.strip()
        if not l or l in repeated:
            continue
        if re.fullmatch(r"\d{1,3}", l):          # page numbers
            continue
        if "INTERNATIONAL MONETARY FUND" in l and len(l) < 40:
            continue
        lines.append(l)
    t = "\n".join(lines)
    t = re.sub(r"-\n(\w)", r"\1", t)          # join hyphenated words
    t = re.sub(r"(?<![.:;!?])\n(?=[a-z])", " ", t)  # join broken sentences
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

clean_pages = [clean_page(p) for p in pages]
full_text = "\n\n".join(clean_pages)
open("outputs/clean_text.txt", "w").write(full_text)
print("Characters after cleaning:", len(full_text))

### 1.3 Token-aware chunking (320 tokens, 50 overlap)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

CHUNK_SIZE, OVERLAP = 320, 50   # 320 (inside the brief's 300-500 range) so every chunk fits the
                                 # 384 word-piece limit of the all-mpnet-base-v2 embedder used in Task 3
                                 # - longer chunks would be silently cut off before embedding

def chunk_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    """Sentence-aware chunking: chunks never cut a sentence in half.
    Sentences are added until ~`size` tokens; the last ~`overlap` tokens of
    sentences are repeated at the start of the next chunk so no context is lost."""
    sentences = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        t = enc.encode(s)
        if len(t) > size:               # tables have no full stops -> split by tokens
            sentences += [enc.decode(t[i:i + size // 2]) for i in range(0, len(t), size // 2)]
        else:
            sentences.append(s)
    chunks, current, cur_len = [], [], 0
    for s in sentences:
        n = len(enc.encode(s))
        if cur_len + n > size and current:
            chunks.append(" ".join(current))
            # keep trailing sentences as overlap
            keep, keep_len = [], 0
            for prev in reversed(current):
                keep_len += len(enc.encode(prev))
                if keep_len > overlap:
                    break
                keep.insert(0, prev)
            current, cur_len = keep, sum(len(enc.encode(x)) for x in keep)
        current.append(s)
        cur_len += n
    if current:
        chunks.append(" ".join(current))
    return chunks

chunks = chunk_text(full_text)
lens = [len(enc.encode(c)) for c in chunks]
print("Total tokens:", len(enc.encode(full_text)))
print("Chunks:", len(chunks), "| tokens per chunk: min", min(lens), "max", max(lens),
      "mean", round(np.mean(lens)))
json.dump(chunks, open("outputs/chunks.json", "w"))
print(chunks[3][:500])

### 1.4 LLM helper functions

In [ ]:
import hashlib
os.makedirs("cache", exist_ok=True)
CACHE_PATH = "cache/gemini_responses.json"

# Saved Gemini answers: download my cache from GitHub so the notebook reproduces
# my results without an API key (and without using any API quota).
if not os.path.exists(CACHE_PATH):
    try:
        r = requests.get(f"{RAW}/gemini_responses.json", timeout=60)
        if r.status_code == 200 and r.text.strip().startswith("{"):
            open(CACHE_PATH, "w").write(r.text)
    except Exception as e:
        print("Cache download failed:", e)
try:
    LLM_CACHE = json.load(open(CACHE_PATH))
except Exception:
    LLM_CACHE = {}
print("Saved LLM answers:", len(LLM_CACHE))

In [ ]:
class LLMUnavailable(RuntimeError):
    """Raised when a prompt is not cached and cannot be sent (no key / daily quota used up)."""

QUOTA_EXHAUSTED = False   # set True after a daily-quota 429 so we stop wasting retries

def ask_llm(prompt, model=EXTRACT_MODEL, system=None, json_mode=False, retries=3):
    """Send one prompt to Gemini and return the text reply.

    - temperature=0 -> answers are repeatable (important for evaluation).
    - Local disk cache: the same model+prompt is never sent twice.
    - Retries with back-off for short rate limits; a *daily* quota error stops at once.
    - Empty replies are never cached, so a failed call can be re-tried later.
    """
    global QUOTA_EXHAUSTED
    key = hashlib.md5(f"{model}|{system}|{json_mode}|{prompt}".encode()).hexdigest()
    if key in LLM_CACHE:
        ask_llm.last_latency = LLM_CACHE[key]["latency"]
        return LLM_CACHE[key]["text"]
    ask_llm.last_latency = None
    if client is None:
        raise LLMUnavailable("No API key and this prompt is not in the saved cache.")
    if QUOTA_EXHAUSTED:
        raise LLMUnavailable("Daily API quota already used up in this session.")
    cfg = {"temperature": 0}
    if system:
        cfg["system_instruction"] = system
    if json_mode:
        cfg["response_mime_type"] = "application/json"   # forces valid JSON output
    for i in range(retries):
        try:
            t0 = time.time()
            r = client.models.generate_content(model=model, contents=prompt, config=cfg)
            ask_llm.last_latency = time.time() - t0
            text = r.text or ""
            if text.strip():                                     # only cache real answers
                LLM_CACHE[key] = {"text": text, "latency": ask_llm.last_latency}
                json.dump(LLM_CACHE, open(CACHE_PATH, "w"))
            return text
        except Exception as e:
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg and "per day" in msg.lower():
                QUOTA_EXHAUSTED = True                           # retrying cannot help today
                raise LLMUnavailable("Daily Gemini quota exhausted - re-run tomorrow.") from e
            print(f"Attempt {i+1} failed: {msg[:150]}")
            time.sleep(15 * (i + 1))
    raise LLMUnavailable("Gemini call failed after retries.")

In [ ]:
def safe_ask(*args, default="", **kwargs):
    """ask_llm that never crashes the notebook: returns `default` if the call is impossible.
    Used for optional/extra steps so the marker can still run end-to-end."""
    try:
        return ask_llm(*args, **kwargs)
    except LLMUnavailable as e:
        print("Skipped LLM call:", e)
        return default

def to_json(text, default=None):
    """Parse LLM text as JSON. Never crashes: returns `default` if parsing fails."""
    default = {} if default is None else default
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}|\[.*\]", text or "", re.S)   # strip ```json fences etc.
        try:
            return json.loads(m.group()) if m else default
        except Exception:
            return default

### 1.5 Executive summary

In [ ]:
SYSTEM_ANALYST = ("You are an IMF financial-stability analyst. Use ONLY the text given. "
                  "Never invent numbers. If something is not in the text, say so.")

# Gemini Flash has a big context window, so we can pass most of the report
doc_for_llm = enc.decode(enc.encode(full_text)[:200000])

exec_prompt = f'''Write 8-10 concise bullet points summarising the main
financial stability findings and risk assessments of this IMF report on Kenya.
Include key numbers where the text gives them.

REPORT:
{doc_for_llm}'''

executive_summary = ask_llm(exec_prompt, system=SYSTEM_ANALYST)
print(executive_summary)

### 1.6 Section summaries (< 100 words each)

In [ ]:
# Main IMF staff-report headings (upper-case lines)
heading_re = re.compile(r"^(?:[A-Z][A-Z ,&\-–'’]{8,80})$", re.M)
KEY = ["CONTEXT", "RECENT DEVELOPMENTS", "OUTLOOK", "RISK", "POLICY DISCUSSIONS",
       "FISCAL", "MONETARY", "FINANCIAL SECTOR", "STRUCTURAL", "PROGRAM", "STAFF APPRAISAL",
       "GOVERNANCE", "CLIMATE", "EXTERNAL", "DEBT"]

found = [(m.start(), m.group().strip()) for m in heading_re.finditer(full_text)
         if any(k in m.group() for k in KEY)]
# keep first occurrence of each heading
seen, heads = set(), []
for pos, h in found:
    if h not in seen:
        seen.add(h); heads.append((pos, h))
print(len(heads), "sections:", [h for _, h in heads])

In [ ]:
sections = {}
for i, (pos, h) in enumerate(heads):
    end = heads[i + 1][0] if i + 1 < len(heads) else len(full_text)
    body = full_text[pos:end]
    if len(body) > 500:                       # skip tiny ones
        sections[h] = enc.decode(enc.encode(body)[:12000])
print("Sections to summarise:", len(sections))

In [ ]:
section_summaries = {}
for h, body in sections.items():
    p = f"Summarise this section of an IMF report in under 100 words.\n\nSECTION: {h}\n{body}"
    section_summaries[h] = ask_llm(p, system=SYSTEM_ANALYST)
    print(f"### {h}\n{section_summaries[h]}\n")

### 1.7 Dual-LLM evaluator (LLM-as-a-Judge)

In [ ]:
JUDGE_SYSTEM = ("You are a strict auditor. Compare the OUTPUT with the SOURCE text. "
                "Score 1-5 for consistency, completeness and factual_alignment. "
                "Return JSON only.")

def judge(output, source, task="summary"):
    """LLM-as-a-Judge. Returns scores, or NaN scores if the output is empty / judge unavailable
    (an empty output must never be scored - that would hide a failure)."""
    empty = {"consistency": None, "completeness": None, "factual_alignment": None}
    if not str(output).strip() or str(output).strip() in ("{}", "[]"):
        return {**empty, "comments": "not judged: empty output"}
    p = f'''TASK: {task}
SOURCE:
{source[:60000]}

OUTPUT:
{output}

Return JSON: {{"consistency": int, "completeness": int,
"factual_alignment": int, "comments": "short reason"}}'''
    reply = safe_ask(p, model=JUDGE_MODEL, system=JUDGE_SYSTEM, json_mode=True)
    return to_json(reply) or {**empty, "comments": "not judged: judge unavailable"}

In [ ]:
judge_scores = {"Executive summary": judge(executive_summary, doc_for_llm, "executive summary")}
for h, s in section_summaries.items():
    judge_scores[h] = judge(s, sections[h], "section summary")

judge_df = pd.DataFrame(judge_scores).T
judge_df

**Is the judge actually strict? (planted-error test).** Every summary scored 5/5, which
could mean the summaries are perfect *or* that the judge agrees with anything. To tell the
difference we give the judge a copy of the executive summary with **deliberately wrong
numbers** and check that its factual-alignment score drops. If it still gives 5/5, the judge
scores above cannot be trusted - an important point for the report.

In [ ]:
# Swap three real figures for wrong ones
PLANTED = {"15.3%": "21.4%", "18.6%": "9.2%", "15.0%": "4.1%"}
corrupted = executive_summary
for real, fake in PLANTED.items():
    corrupted = corrupted.replace(real, fake)
print("Numbers changed:", sum(real in executive_summary for real in PLANTED))

In [ ]:
sanity = pd.DataFrame({
    "original":  judge(executive_summary, doc_for_llm, "executive summary"),
    "corrupted": judge(corrupted, doc_for_llm, "executive summary"),
}).T
json.dump(sanity.to_dict(orient="index"), open("outputs/judge_sanity_check.json", "w"), indent=2)
sanity

## Task 2 — Themes, indicators & structured output
### 2.1 Thematic distribution

In [ ]:
# Keywords per theme. Overlapping phrases were removed so a phrase is not counted twice
# (e.g. "debt service" was also matched by "debt"; "consumer prices" also by "price").
THEMES = {
    "Inflation & Prices": ["inflation", "price", "prices", "cpi"],
    "Interest Rates & Credit": ["interest rate", "policy rate", "credit", "lending", "cbr"],
    "Housing & Real Estate": ["housing", "real estate", "mortgage", "property"],
    "Banking Capital & Liquidity": ["capital adequacy", "liquidity", "bank capital", "tier 1", "buffer"],
    "Sovereign Debt": ["debt", "eurobond", "borrowing", "fiscal deficit"],
    "Labor Market Dynamics": ["employment", "unemployment", "labor", "labour", "wage", "jobs"],
    "Systemic Financial Risk": ["systemic", "financial stability", "npl", "non-performing", "contagion", "stress test"],
}

def count_theme(text, kws):
    """Whole-word keyword count (same rule for the full text and for each chunk)."""
    return sum(len(re.findall(r"\b" + re.escape(k) + r"\b", text)) for k in kws)

In [ ]:
low = full_text.lower()
theme_counts = {t: count_theme(low, kws) for t, kws in THEMES.items()}

# Which theme dominates each chunk ("Other" = no theme keyword at all)
chunk_theme = []
for c in chunks:
    scores = {t: count_theme(c.lower(), kws) for t, kws in THEMES.items()}
    chunk_theme.append(max(scores, key=scores.get) if max(scores.values()) > 0 else "Other")

theme_df = pd.DataFrame({"mentions": pd.Series(theme_counts),
                         "chunks_dominant": pd.Series(chunk_theme).value_counts()}).fillna(0)
theme_df["share_pct"] = (100 * theme_df["mentions"] / theme_df["mentions"].sum()).round(1)
json.dump({"mentions": theme_counts,
           "chunks_dominant": theme_df["chunks_dominant"].astype(int).to_dict()},
          open("outputs/theme_counts.json", "w"), indent=2)
theme_df.sort_values("mentions", ascending=False)

### 2.2 Qualitative risk insights

In [ ]:
qual_prompt = f'''From this IMF report on Kenya return JSON:
{{"growth_drivers": [5-7 short items], "vulnerabilities": [5-7 short items]}}

REPORT:
{doc_for_llm}'''
qualitative = to_json(ask_llm(qual_prompt, system=SYSTEM_ANALYST, json_mode=True))
json.dump(qualitative, open("outputs/qualitative_insights.json", "w"), indent=2)
qualitative

### 2.3 Core indicator JSON

In [ ]:
SCHEMA = {
    "country_name": "string",
    "gdp_growth_forecast_pct": "float",
    "headline_inflation_rate_pct": "float",
    "central_bank_policy_rate_pct": "float",
    "non_performing_loan_ratio": "float",
    "unemployment_rate_pct": "float",
    "tier_1_capital_ratio": "float",
}

ind_prompt = f'''Extract these indicators for Kenya from the report.
Use the most recent value / the forecast for the next year.
If a value is NOT stated in the text, use null. Do not guess.
Return ONLY JSON with exactly these keys: {json.dumps(SCHEMA)}
Also add a key "evidence" with a short quote for each number.

REPORT:
{doc_for_llm}'''
raw_ind = to_json(ask_llm(ind_prompt, system=SYSTEM_ANALYST, json_mode=True))
raw_ind

In [ ]:
FALLBACK = -1.0   # clean numeric default: keeps the column a float, easy to spot in plots

def to_pct(v):
    """Convert to float %. Fixes scale errors: a ratio like 0.065 becomes 6.5."""
    v = float(str(v).replace("%", "").strip())
    return round(v * 100, 2) if 0 < abs(v) < 0.2 else v

def validate_indicators(raw, schema):
    """Enforce the schema: right keys, right types, fallback + reason if missing."""
    out, log = {}, {}
    for k, typ in schema.items():
        v = raw.get(k)
        if typ == "string":
            out[k] = v if isinstance(v, str) and v else "Kenya"
            continue
        try:
            out[k] = to_pct(v)
            log[k] = "extracted"
        except (TypeError, ValueError):
            out[k] = FALLBACK
            log[k] = "fallback: not stated in report" if v is None else f"fallback: unparseable '{v}'"
    return out, log

indicators, extraction_log = validate_indicators(raw_ind, SCHEMA)
json.dump(indicators, open("outputs/core_indicators.json", "w"), indent=2)
json.dump({"log": extraction_log, "evidence": raw_ind.get("evidence", {})},
          open("outputs/extraction_log.json", "w"), indent=2)
pd.DataFrame({"value": indicators, "status": extraction_log})

### 2.4 Judge audits the JSON numbers

In [ ]:
audit_prompt = f'''Check each number in EXTRACTED against the SOURCE.
Return JSON: {{"<indicator>": {{"correct": true/false, "comment": "..."}}, ...,
"overall_accuracy_score": 1-5}}

EXTRACTED: {json.dumps(raw_ind)}
SOURCE:
{doc_for_llm[:150000]}'''
json_audit = to_json(ask_llm(audit_prompt, model=JUDGE_MODEL, system=JUDGE_SYSTEM, json_mode=True))
json.dump(json_audit, open("outputs/indicator_audit.json", "w"), indent=2)
json_audit

**Non-LLM cross-check.** The judge is itself an LLM and can be wrong (e.g. agreeing too easily).
So we also check in plain Python whether each extracted number really appears in the report text,
and compare that with the judge's verdict. Disagreements are worth discussing in the report.

In [ ]:
def number_in_text(v, text):
    if v == FALLBACK:
        return None
    forms = {f"{v:.1f}", f"{v:g}", f"{v:.2f}"}
    return any(re.search(r"(?<![\d.])" + re.escape(f) + r"(?![\d])", text) for f in forms)

check = pd.DataFrame({
    "value": {k: v for k, v in indicators.items() if k != "country_name"},
})
check["found_in_pdf_text"] = [number_in_text(v, full_text) for v in check["value"]]
check["judge_says_correct"] = [json_audit.get(k, {}).get("correct") if isinstance(json_audit.get(k), dict)
                               else None for k in check.index]
check.to_json("outputs/indicator_crosscheck.json", indent=2)
check

### 2.5 Temporal trends (multi-year series)

In [ ]:
trend_prompt = f'''From the report's tables (e.g. Selected Economic Indicators), extract
year-by-year values for: real_gdp_growth, inflation, policy_rate, public_debt_pct_gdp,
private_credit_growth, npl_ratio. Include historical and projected years.
Return JSON: {{"years": [..], "real_gdp_growth": [..], "inflation": [..],
"policy_rate": [..], "public_debt_pct_gdp": [..], "private_credit_growth": [..],
"npl_ratio": [..]}}  Use null where a year is missing. Lists must be the same length.

REPORT:
{doc_for_llm}'''
trends = to_json(ask_llm(trend_prompt, system=SYSTEM_ANALYST, json_mode=True))
json.dump(trends, open("outputs/temporal_trends.json", "w"), indent=2)

In [ ]:
def clean_trends(trends):
    """Make a tidy numeric table: equal-length lists, numbers as floats, and a
    whole-series scale fix (if every value is < 1 it was given as a ratio)."""
    years = trends.get("years", [])
    df = pd.DataFrame({"years": [str(y) for y in years]})
    for k, vals in trends.items():
        if k == "years" or not isinstance(vals, list):
            continue
        vals = (vals + [None] * len(years))[:len(years)]      # pad / trim to same length
        s = pd.to_numeric(pd.Series(vals).astype(str).str.replace("%", ""), errors="coerce")
        if s.notna().any() and s.abs().max() < 1:
            s = s * 100
        df[k] = s.values
    return df

trend_df = clean_trends(trends)
trend_df

In [ ]:
# Save everything from Tasks 1-2
json.dump({"executive_summary": executive_summary,
           "section_summaries": section_summaries,
           "judge_scores": judge_scores},
          open("outputs/summaries.json", "w"), indent=2)
print(os.listdir("outputs"))

## Task 3 — RAG grounding
### 3.1 Embeddings + FAISS index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, torch

# all-mpnet-base-v2: strong general-purpose sentence-transformers model (768-dim).
# It is slow on CPU (~30 s per batch), so: use the GPU when Colab has one
# (Runtime -> Change runtime type -> T4 GPU) and save the vectors to disk so it runs once.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("all-mpnet-base-v2", device=DEVICE)
print("Embedding on:", DEVICE, "| max tokens:", embedder.max_seq_length)

In [ ]:
EMB_PATH = "outputs/embeddings.npy"
if not os.path.exists(EMB_PATH):                      # try the copy saved in the repo first
    try:
        r = requests.get(f"{RAW}/embeddings.npy", timeout=60)
        if r.status_code == 200:
            open(EMB_PATH, "wb").write(r.content)
    except Exception:
        pass

emb = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None
if emb is None or len(emb) != len(chunks):            # recompute if missing / out of date
    # normalised vectors -> L2 distance ranks exactly like cosine similarity
    emb = embedder.encode(chunks, batch_size=64, show_progress_bar=True,
                          normalize_embeddings=True).astype("float32")
    np.save(EMB_PATH, emb)
print("Embeddings:", emb.shape)

In [ ]:
index = faiss.IndexFlatL2(emb.shape[1])
index.add(emb)
print("Vectors in index:", index.ntotal)

### 3.2 Retrieve + grounded answer

In [ ]:
def retrieve(query, k=4):
    """Return the k chunks closest to the query (same normalisation as the index)."""
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, ids = index.search(q, k)
    return [chunks[i] for i in ids[0]]

def rag_answer(query, ctx):
    p = f'''Answer using ONLY the context. If the answer is not there, say "Not in report".
CONTEXT:
{chr(10).join(ctx)}

QUESTION: {query}'''
    return safe_ask(p, system=SYSTEM_ANALYST)

In [ ]:
RAG_QUESTIONS = [
    "What does the IMF recommend about bank capital buffers in Kenya?",
    "What is the Central Bank of Kenya's inflation target range?",
    "What are the main risks to public debt sustainability?",
    "What assumptions are used in the banking stress tests?",
]
rag_results = []
for q in RAG_QUESTIONS:
    ctx = retrieve(q)
    ans = rag_answer(q, ctx)
    score = judge(ans, "\n".join(ctx), "RAG answer")
    rag_results.append({"question": q, "answer": ans, **score})
    print("Q:", q, "\nA:", ans, "\n")

json.dump(rag_results, open("outputs/rag_results.json", "w"), indent=2)

## Task 4 — Dashboard
### 4.1 Tidy (long) data with pd.melt

In [ ]:
trend_long = trend_df.melt(id_vars="years", var_name="indicator", value_name="value").dropna()

judge_long = (judge_df.reset_index().rename(columns={"index": "output"})
              .melt(id_vars="output", value_vars=["consistency", "completeness", "factual_alignment"],
                    var_name="metric", value_name="score"))
judge_long["score"] = pd.to_numeric(judge_long["score"], errors="coerce")

rag_df = pd.DataFrame(rag_results)
rag_long = rag_df.melt(id_vars="question", value_vars=["consistency", "completeness", "factual_alignment"],
                       var_name="metric", value_name="score")
rag_long["score"] = pd.to_numeric(rag_long["score"], errors="coerce")

ind_long = (pd.DataFrame([indicators]).melt(id_vars="country_name",
            var_name="indicator", value_name="value"))
trend_long.head()

### 4.2 Four-plot dashboard (300 dpi)
Each panel is a small function; the last cell assembles and saves them.

In [ ]:
# Each panel is its own function, so the whole dashboard is drawn and saved in ONE cell
# (a matplotlib figure split across two Colab cells is shown half-empty).
plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

def plot_themes(ax):
    t = theme_df.drop(index="Other", errors="ignore")["mentions"].sort_values()   # 7 themes only
    ax.barh(t.index, t.values, color="#2b6cb0")
    for y, v in enumerate(t.values):
        ax.text(v, y, f" {int(v)}", va="center", fontsize=9)
    ax.set_title("Thematic density – 7 financial stability themes")
    ax.set_xlabel("Keyword mentions")

def plot_trends(ax):
    labels = {"inflation": "Headline inflation", "policy_rate": "CBK policy rate",
              "real_gdp_growth": "Real GDP growth"}
    for col, lab in labels.items():
        if col in trend_df:
            ax.plot(trend_df["years"].astype(str), pd.to_numeric(trend_df[col], errors="coerce"),
                    marker="o", label=lab)
    if "2024" in set(trend_df["years"].astype(str)):
        ax.axvspan("2024", trend_df["years"].astype(str).iloc[-1], color="grey", alpha=0.1,
                   label="Projection")
    ax.set_title("Inflation vs policy rate vs GDP growth")
    ax.set_ylabel("Percent (%)"); ax.legend(); ax.tick_params(axis="x", rotation=45)

In [ ]:
def plot_judge(ax):
    """Summary judge vs RAG judge, side by side per metric."""
    comp = pd.DataFrame({"Summaries": judge_long.groupby("metric")["score"].mean(),
                         "RAG answers": rag_long.groupby("metric")["score"].mean()})
    comp.plot(kind="bar", ax=ax, rot=0, color=["#38a169", "#dd6b20"])
    ax.set_ylim(0, 5.3); ax.set_ylabel("Mean score (1–5)")
    ax.set_title(f"LLM-as-a-Judge mean scores ({JUDGE_MODEL})")

def plot_credit_npl(ax):
    x = pd.to_numeric(trend_df.get("private_credit_growth"), errors="coerce")
    y = pd.to_numeric(trend_df.get("npl_ratio"), errors="coerce")
    ok = x.notna() & y.notna()
    if ok.any():
        ax.scatter(x[ok], y[ok], s=80, color="#c53030", label="Year (reported)")
        for xi, yi, yr in zip(x[ok], y[ok], trend_df["years"][ok]):
            ax.annotate(str(yr), (xi, yi), textcoords="offset points", xytext=(5, 5))
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Credit growth / NPL series not reported\n(see fallback log)",
                ha="center", va="center", transform=ax.transAxes, color="grey")
    ax.set_title(f"Private credit growth vs NPL ratio (n = {int(ok.sum())} years)")
    ax.set_xlabel("Credit growth (%)"); ax.set_ylabel("NPL ratio (%)")

In [ ]:
def fmt(k):
    v = indicators[k]
    return "n/a (fallback)" if v == FALLBACK else f"{v:.1f}%"

fig, ax = plt.subplots(2, 2, figsize=(16, 11))
plot_themes(ax[0, 0]); plot_trends(ax[0, 1]); plot_judge(ax[1, 0]); plot_credit_npl(ax[1, 1])
fig.suptitle("Kenya – IMF Country Report 24/13 Dashboard", fontsize=16, weight="bold")

overlay = (f"GDP growth forecast: {fmt('gdp_growth_forecast_pct')}   |   "
           f"Inflation: {fmt('headline_inflation_rate_pct')}   |   "
           f"Policy rate: {fmt('central_bank_policy_rate_pct')}   |   "
           f"NPL ratio: {fmt('non_performing_loan_ratio')}\n"
           f"Top theme: {theme_df.drop(index='Other', errors='ignore')['mentions'].idxmax()}   |   "
           f"Mean judge score: {judge_long['score'].mean():.2f} / 5")
fig.text(0.5, 0.01, overlay, ha="center", va="bottom", fontsize=11,
         bbox=dict(boxstyle="round", facecolor="#f7fafc", edgecolor="#a0aec0"))
fig.tight_layout(rect=[0, 0.07, 1, 0.96])
fig.savefig("figures/dashboard.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Extra plot: extracted core indicators (fallbacks shown in grey)
d = ind_long.copy()
colors = ["grey" if v == FALLBACK else "#2b6cb0" for v in d["value"]]
plt.figure(figsize=(10, 5))
plt.barh(d["indicator"], d["value"], color=colors)
plt.title("Core indicators extracted (grey = fallback -1.0)")
plt.xlabel("%"); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("figures/core_indicators.png", dpi=300)
plt.show()

## Extension — System-instruction directives vs generic prompts
The same extraction tasks are run two ways:
- **Strict:** a `system_instruction` that sets the analyst role and rules, plus Gemini's JSON mode.
- **Generic:** the task only, as a plain user prompt (no system instruction, no JSON mode).

Compared on **schema compliance** (valid JSON, all keys, correct types), **latency**, and the **judge's accuracy score**.

In [ ]:
GENERIC_TASKS = {
    "indicators": f"Give me these numbers for Kenya from the report as JSON with keys "
                  f"{list(SCHEMA)}.\n\nREPORT:\n{doc_for_llm}",
    "insights":   f"List Kenya's growth drivers and vulnerabilities from this report as JSON with keys "
                  f"growth_drivers and vulnerabilities.\n\nREPORT:\n{doc_for_llm}",
}
STRICT_TASKS = {"indicators": ind_prompt, "insights": qual_prompt}
REQUIRED = {"indicators": list(SCHEMA), "insights": ["growth_drivers", "vulnerabilities"]}

def schema_check(text, keys):
    """Score one reply: is it valid JSON, are all keys there, are the numbers numbers?"""
    try:
        obj = json.loads(text)
        valid = True
    except Exception:
        obj, valid = to_json(text), False          # rescued only by regex = not compliant
    present = sum(k in obj for k in keys) / len(keys)
    numeric_ok = all(isinstance(obj.get(k), (int, float)) or obj.get(k) is None
                     for k in keys if SCHEMA.get(k) == "float")
    return obj, {"valid_json": valid, "keys_present": round(present, 2), "types_ok": numeric_ok}

In [ ]:
bench = []
for mode, tasks in [("strict", STRICT_TASKS), ("generic", GENERIC_TASKS)]:
    for task, prompt in tasks.items():
        if mode == "strict":
            text = safe_ask(prompt, system=SYSTEM_ANALYST, json_mode=True)
        else:
            text = safe_ask(prompt)                       # no system instruction, no JSON mode
        latency = ask_llm.last_latency
        row = {"mode": mode, "task": task, "status": "ok" if text.strip() else "no reply (quota / no key)",
               "latency_s": round(latency, 2) if latency else None}
        if text.strip():                                  # only score replies that exist
            obj, checks = schema_check(text, REQUIRED[task])
            verdict = judge(json.dumps(obj)[:4000], doc_for_llm, f"{task} extraction")
            row.update(checks, judge_accuracy=verdict.get("factual_alignment"))
        bench.append(row)

bench_df = pd.DataFrame(bench)
bench_df.to_json("outputs/directive_benchmark.json", orient="records", indent=2)
bench_df

In [ ]:
done = bench_df[bench_df["status"] == "ok"].copy()        # failed calls are reported, not plotted as 0
if done["mode"].nunique() < 2:
    print("Benchmark incomplete - only these modes have replies:", list(done["mode"].unique()),
          "\nRe-run the cell above when the API quota resets.")
else:
    plot_df = (done.assign(valid_json=done["valid_json"].astype(int), types_ok=done["types_ok"].astype(int))
               .melt(id_vars=["mode"], value_vars=["valid_json", "keys_present", "types_ok", "judge_accuracy"],
                     var_name="metric", value_name="value")
               .assign(value=lambda d: pd.to_numeric(d["value"], errors="coerce"))
               .groupby(["metric", "mode"])["value"].mean().unstack())

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    plot_df.loc[["valid_json", "keys_present", "types_ok"]].plot(kind="bar", ax=ax[0], rot=0, ylim=(0, 1.05))
    ax[0].set_title("Schema compliance (share of replies)"); ax[0].set_ylabel("0 – 1")
    lat = done.groupby("mode")["latency_s"].mean()
    ax[1].bar(lat.index, lat.values, color=["#805ad5", "#38a169"])
    ax[1].set_title("Mean extraction latency"); ax[1].set_ylabel("Seconds")
    for a in ax: a.grid(alpha=0.3)
    fig.suptitle("System-instruction directive vs generic prompt", weight="bold")
    fig.tight_layout(); fig.savefig("figures/directive_benchmark.png", dpi=300)
    plt.show()
    print(plot_df.round(2))

## Download all outputs

In [ ]:
# Zip the cache, embeddings, outputs and figures. Upload gemini_responses.json and
# embeddings.npy to the GitHub repo so the marker's run needs no key and no slow embedding.
import shutil
shutil.copy(CACHE_PATH, "outputs/gemini_responses.json")
shutil.make_archive("pipeline_outputs", "zip", ".", "outputs")
shutil.make_archive("pipeline_figures", "zip", ".", "figures")
print("Saved answers:", len(LLM_CACHE))
try:
    from google.colab import files
    for f in ["pipeline_outputs.zip", "pipeline_figures.zip"]:
        files.download(f)
except ImportError:
    print("Not in Colab - zips are in the working folder.")